<a href="https://colab.research.google.com/github/noorulaein/urdu-ocr-codesaviours-si26-noorulaein-fatima-/blob/main/SI26_Week4_noorulaein_fatima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y transformers tokenizers sentencepiece

!pip install transformers==4.41.2
!pip install tokenizers==0.19.1
!pip install sentencepiece==0.2.0
!pip install protobuf==4.25.3

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: tokenizers 0.19.1
Uninstalling tokenizers-0.19.1:
  Successfully uninstalled tokenizers-0.19.1
Found existing installation: sentencepiece 0.2.0
Uninstalling sentencepiece-0.2.0:
  Successfully uninstalled sentencepiece-0.2.0
  Using cached transformers-4.41.2-py3-none-any.whl.metadata (43 kB)
  Using cached tokenizers-0.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.41.2-py3-none-any.whl (9.1 MB)
Using cached tokenizers-0.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtune 0.6.1 requires sentencepiece, which is not installed.


  Using cached sentencepiece-0.2.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.7 kB)
Using cached sentencepiece-0.2.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (1.3 MB)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
^C


In [1]:
import os
import cv2
import torch
import pandas as pd

from PIL import Image
from sklearn.model_selection import train_test_split

from transformers import TrOCRProcessor
from transformers import VisionEncoderDecoderModel

from torch.utils.data import Dataset, DataLoader

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [3]:
processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)

model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
)

model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_fea

In [19]:
# Configure model
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.vocab_size = model.config.decoder.vocab_size

In [20]:
print("decoder_start_token_id:", model.config.decoder_start_token_id)
print("pad_token_id:", model.config.pad_token_id)
print("eos_token_id:", model.config.eos_token_id)

decoder_start_token_id: 0
pad_token_id: 1
eos_token_id: 2


In [4]:
csv_path="/content/drive/MyDrive/data/updated_labels.csv"

df=pd.read_csv(csv_path)

df.head()

,image,text,metadata
0,/content/drive/MyDrive/data/original/001_001_1...,کون سوچ سکتا تھا کہ ہندوستان اکثریت اور انگریز...,NaN
1,/content/drive/MyDrive/data/original/001_001_2...,مخالفت کے باوجود برصغیر کی ملتِ اسلامیہ دینِ ا...,NaN
2,/content/drive/MyDrive/data/original/001_001_3...,پر اس ملک میں بسنے والے مختلف عناصر کا اتحاد ہ...,NaN
3,/content/drive/MyDrive/data/original/001_001_4...,بقاً اسی نظریہ حیات کے فروغ پر منحصر ہے۔,NaN
4,/content/drive/MyDrive/data/original/001_001_5...,لیکن بدقسمتی سے پاکستان بننے کے بعد ہی اس کے ا...,NaN


In [5]:
train_df,test_df=train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

In [13]:
from torch.utils.data import Dataset
from PIL import Image

class UrduOCRDataset(Dataset):
    def __init__(self, dataframe, processor):
        self.data = dataframe.reset_index(drop=True)
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        # Get image path directly from CSV
        image_path = self.data.loc[idx, "image"]

        # Open image
        image = Image.open(image_path).convert("RGB")

        # Process image
        pixel_values = self.processor(
            image,
            return_tensors="pt"
        ).pixel_values.squeeze(0)

        # Get Urdu text
        text = str(self.data.loc[idx, "text"])

        # Tokenize text
        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).input_ids.squeeze(0)

        # Ignore padding in loss
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

In [14]:
train_dataset = UrduOCRDataset(train_df, processor)
test_dataset = UrduOCRDataset(test_df, processor)

print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))

Training images: 175
Testing images: 44


In [15]:
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

In [16]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=5e-5
)

In [21]:
num_epochs=3

for epoch in range(num_epochs):

    model.train()

    total_loss=0

    print(f"\nEpoch {epoch+1}/{num_epochs}")

    for batch in train_loader:

        pixel_values=batch["pixel_values"].to(device)

        labels=batch["labels"].to(device)

        outputs=model(
            pixel_values=pixel_values,
            labels=labels
        )

        loss=outputs.loss

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss+=loss.item()

    avg_loss=total_loss/len(train_loader)

    print(avg_loss)


Epoch 1/3
5.238897984678095

Epoch 2/3
3.577730829065496

Epoch 3/3
3.4761282964186235


In [23]:
model.eval()

print("===== Model Evaluation =====\n")

correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]

        # Generate predictions
        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=128
        )

        predicted_text = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        # Replace -100 with pad token before decoding
        labels = labels.clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id

        actual_text = processor.batch_decode(
            labels,
            skip_special_tokens=True
        )

        for pred, actual in zip(predicted_text, actual_text):

            print("Predicted :", pred)
            print("Actual    :", actual)
            print("-" * 60)

            total += 1

            if pred.strip() == actual.strip():
                correct += 1

accuracy = (correct / total) * 100

print(f"\nAccuracy: {accuracy:.2f}%")
print(f"Correct : {correct}")
print(f"Total   : {total}")

===== Model Evaluation =====

Predicted : �������������������������������������������������������������������������������������������������������������������������������
Actual    : مظاہرین میں سے ایک شخص پولیس کی فائرنگ سے ہلاک
------------------------------------------------------------
Predicted : �������������������������������������������������������������������������������������������������������������������������������
Actual    : کی نیلامی کی جاتی ہے اور بچے جاتے ہیں، مسٹر
------------------------------------------------------------
Predicted : �������������������������������������������������������������������������������������������������������������������������������
Actual    : وقت ضائع نہ کرو
------------------------------------------------------------
Predicted : �������������������������������������������������������������������������������������������������������������������������������
Actual    : خوش رہو اور خوشیاں بانٹو
-----------------------------------------------

In [24]:
save_path = "/content/drive/MyDrive/SI26-urdu-ocr-model"

model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print("Model saved successfully!")
print(save_path)

Model saved successfully!
/content/drive/MyDrive/SI26-urdu-ocr-model
